# 摘要记忆

> **用 LLM 生成的滚动摘要压缩较早的对话轮次：用逐字回忆换取无限对话长度。**

在[上一份 notebook](../02_sliding_window_memory/sliding_window_memory.ipynb)中我们看到**滑动窗口记忆**通过完全丢弃旧消息来限制成本。这种硬截断意味着 Agent 甚至*不知道*它遗忘了什么。

**摘要记忆**采用不同的方法。它不是丢弃历史，而是*压缩*历史。想象阅读一本长书并在每章结尾写一页摘要。你不能再逐字逐句引用原书，但你仍然知道发生了什么。一个辅助 LLM 调用会定期将旧消息浓缩为运行中的文本摘要。Agent 失去了精确措辞，但能在任意长的对话中保留大意（关键事实、决策和上下文）。

**代价：** 摘要是有损的（它们无法完美重建原文）。每次压缩循环都可能丢失细节、转移重点或微妙地扭曲事实。经过多次循环后，这种**摘要漂移**会累积。Agent 的"记忆"可能与实际发生的情况产生偏差。

**完成本 notebook 后你将理解：**
- 如何使用 Anthropic SDK 从头构建滚动摘要记忆系统。
- 摘要循环：何时触发、使用什么提示词，以及摘要如何演化。
- 摘要如何在长对话中漂移，附有对照实验和可视化。
- 针对漂移和信息丢失的实用缓解措施。

## 核心概念

- **滚动摘要**：一个单一的文本块，随着对话的增长而增量更新。每次更新将新消息合并到现有摘要中。
- **摘要提示词**：给 LLM 生成摘要的指令。其措辞控制保留什么（事实、决策、语气）和丢弃什么。
- **刷新触发器**：决定*何时*重新摘要的规则。常见选项：每 *n* 条消息后、缓冲区超过 token 阈值时，或每轮都刷新。
- **摘要漂移**：事实在重复摘要周期中逐渐失真。细节会随着时间的推移被软化、合并或完全丢失。
- **压缩比**：与所替代的原始消息相比，摘要缩短了多少。更高的压缩意味着更多的信息丢失。
- **缓冲区**：在摘要旁按原文保留的最近消息。这为 LLM 提供了最新对话内容的精确上下文，而较早的上下文则存在于压缩摘要中。

## 架构

<p align="center">
  <img src="../../images/diagrams/03_summary_memory.svg" alt="diagram" width="720"/>
</p>

<details><summary>Mermaid 源代码</summary>

```mermaid
sequenceDiagram
    participant U as 用户
    participant B as 消息缓冲区
    participant S as 摘要存储
    participant LLM as Claude (聊天)
    participant SLLM as Claude (摘要器)

    Note over S: 摘要: ""（空）
    Note over B: 缓冲区: []

    U->>B: msg 1
    B->>LLM: [summary="", msg1]
    LLM-->>B: msg 2
    Note over B: 缓冲区: [msg1, msg2]

    U->>B: msg 3
    B->>LLM: [summary="", msg1..msg3]
    LLM-->>B: msg 4
    Note over B: 缓冲区: [msg1..msg4] - 触发！

    rect rgb(255, 245, 230)
        Note over B,SLLM: 🔄 摘要周期
        B->>SLLM: "摘要: {old_summary} + {msg1..msg4}"
        SLLM-->>S: 更新的摘要
        B->>B: 清空缓冲区
    end

    U->>B: msg 5
    B->>LLM: [summary="...", msg5]
    LLM-->>B: msg 6
    Note over B: 缓冲区: [msg5, msg6]
    Note over S: 摘要向前传递<br/>压缩后的历史
```

</details>

In [ ]:
# 安装所需包（运行一次）
%pip install -q anthropic python-dotenv matplotlib pandas numpy

## 环境准备

导入 Anthropic SDK 并从 `.env` 文件加载你的 API 密钥。

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # 从 .env 文件中读取 ANTHROPIC_API_KEY

import anthropic

assert os.getenv("ANTHROPIC_API_KEY"), "请在 .env 文件中设置 ANTHROPIC_API_KEY"
print("✓ Anthropic API 密钥已加载")

## 核心实现

设计分为两部分：

1. **缓冲区**：按原文存储的最近消息（字典列表）。
2. **摘要**：一个压缩所有旧消息的字符串。

当缓冲区达到 `max_buffer_size` 条消息时，我们用*摘要提示词*调用 LLM。该提示词将当前摘要 + 缓冲区合并为一个新摘要。然后我们清空缓冲区。

每轮聊天 LLM 接收：`[system: summary_context] + buffer_messages`。

In [ ]:
SUMMARIZER_PROMPT = """你是一个对话摘要器。根据现有摘要和新消息，
生成一个更新后的摘要，要求：
1. 保留所有关键事实（名称、数字、偏好、决策）。
2. 记录任何未解决的问题或待定话题。
3. 保持简洁（目标是 2-5 句话）。

当前摘要：
{summary}

新消息：
{messages}

只输出更新后的摘要，不要输出其他内容。"""




现在我们定义 `SummaryMemory` 类。构造函数设置两个存储区域：一个 `summary` 字符串（初始为空）和一个 `buffer` 列表用于最近消息。`_summarize` 方法调用 LLM 将当前摘要加上缓冲区压缩为一个更短的新摘要。

In [ ]:
class SummaryMemory:
    """滚动摘要记忆，通过 LLM 压缩较早的对话轮次。"""

    def __init__(
        self,
        max_buffer_size: int = 6,
        model: str = "claude-sonnet-4-20250514",
        summarizer_model: str = "claude-sonnet-4-20250514",
        system_prompt: str | None = None,
        max_tokens: int = 1024,
    ):
        self.client = anthropic.Anthropic()
        self.model = model
        self.summarizer_model = summarizer_model
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        self.max_buffer_size = max_buffer_size

        # 状态
        self.summary: str = ""
        self.buffer: list[dict] = []

        # 追踪
        self.full_history: list[dict] = []
        self.summary_history: list[str] = []  # 每个摘要版本
        self.turn_token_usage: list[dict] = []
        self._summary_calls = 0

    # ── 摘要 ────────────────────────────────────────────
    def _format_messages_for_summary(self, messages: list[dict]) -> str:
        lines = []
        for msg in messages:
            role = "User" if msg["role"] == "user" else "Assistant"
            lines.append(f"{role}: {msg['content']}")
        return "\n".join(lines)

    def _summarize(self) -> str:
        """将当前摘要 + 缓冲区压缩为一个新摘要。"""
        messages_text = self._format_messages_for_summary(self.buffer)
        prompt = SUMMARIZER_PROMPT.format(
            summary=self.summary or "(无先前摘要)",
            messages=messages_text,
        )
        response = self.client.messages.create(
            model=self.summarizer_model,
            max_tokens=512,
            messages=[{"role": "user", "content": prompt}],
        )
        self._summary_calls += 1
        return response.content[0].text.strip()


`chat` 方法是魔法发生的地方。它构建一个包含运行摘要的系统提示词，将缓冲区发送给 LLM，并检查缓冲区是否已超过 `max_buffer_size`。如果是，则触发摘要周期并清空缓冲区。

In [ ]:
    # ── 聊天 ─────────────────────────────────────────────────────
    def chat(self, user_input: str) -> str:
        user_msg = {"role": "user", "content": user_input}
        self.buffer.append(user_msg)
        self.full_history.append(user_msg)

        # 构建包含摘要上下文的系统提示词
        system_parts = []
        if self.system_prompt:
            system_parts.append(self.system_prompt)
        if self.summary:
            system_parts.append(f"当前对话摘要：\n{self.summary}")
        system = "\n\n".join(system_parts) if system_parts else None

        kwargs = dict(
            model=self.model,
            max_tokens=self.max_tokens,
            messages=self.buffer,
        )
        if system:
            kwargs["system"] = system

        response = self.client.messages.create(**kwargs)
        assistant_text = response.content[0].text

        assistant_msg = {"role": "assistant", "content": assistant_text}
        self.buffer.append(assistant_msg)
        self.full_history.append(assistant_msg)

        self.turn_token_usage.append({
            "turn": len(self.turn_token_usage) + 1,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "buffer_msgs": len(self.buffer),
            "summary_len": len(self.summary),
        })

        # 检查是否需要摘要
        if len(self.buffer) >= self.max_buffer_size:
            self.summary = self._summarize()
            self.summary_history.append(self.summary)
            self.buffer.clear()

        return assistant_text


最后，我们添加检查和工具方法。`get_context_snapshot` 精确显示 LLM 在下一次调用时会看到什么：摘要文本加上当前缓冲区内容。

In [ ]:
    # ── 检查 ───────────────────────────────────────────────
    def get_context_snapshot(self) -> dict:
        """返回 LLM 当前看到的内容。"""
        return {
            "summary": self.summary,
            "buffer": list(self.buffer),
            "buffer_size": len(self.buffer),
            "total_messages": len(self.full_history),
            "summary_versions": len(self.summary_history),
        }

    def clear(self) -> None:
        self.summary = ""
        self.buffer.clear()
        self.full_history.clear()
        self.summary_history.clear()
        self.turn_token_usage.clear()
        self._summary_calls = 0

    def __repr__(self) -> str:
        return (
            f"SummaryMemory(buffer={len(self.buffer)}/{self.max_buffer_size}, "
            f"summary_versions={len(self.summary_history)}, "
            f"total_msgs={len(self.full_history)})"
        )


print("✓ SummaryMemory 类已定义")

## 使用示例：观察摘要演化

我们使用一个小缓冲区（`max_buffer_size=4`，意味着 2 轮触发一次摘要）。我们将植入几个事实，观察它们如何被压缩到摘要中。

In [ ]:
mem = SummaryMemory(
    max_buffer_size=4,  # 每 2 轮触发一次摘要
    system_prompt="你是一个简洁的助手。用 1-2 句话回复。",
)

conversation = [
    "我叫 Carlos，来自布宜诺斯艾利斯。",
    "我是一名研究珊瑚礁的海洋生物学家。",
    "我最喜欢的编程语言是 Rust。",
    "目前你对我了解多少？",
]

for msg in conversation:
    print(f"👤 用户:  {msg}")
    reply = mem.chat(msg)
    print(f"🤖 Agent: {reply}")
    snap = mem.get_context_snapshot()
    print(f"   📊 缓冲区: {snap['buffer_size']}/{mem.max_buffer_size} | 摘要版本: {snap['summary_versions']}")
    if snap["summary"]:
        print(f"   📝 当前摘要: {snap['summary'][:120]}...")
    print()

让我们追踪摘要如何随时间演化。每个版本显示摘要器在合并一批消息后产生的内容。我们还打印当前缓冲区内容。

In [ ]:
print("=== 摘要演化 ===\n")
for i, s in enumerate(mem.summary_history):
    print(f"版本 {i+1}:")
    print(f"  {s}")
    print()

print(f"=== 当前缓冲区（{len(mem.buffer)} 条消息）===")
for msg in mem.buffer:
    role = "用户" if msg["role"] == "user" else "助手"
    print(f"  {role}: {msg['content'][:80]}")

## 实验：长对话中的摘要漂移

摘要漂移是本技术最重要的失败模式。每个摘要周期都是有损的，且错误会累积。第 1 轮清晰陈述的事实可能在摘要 v2 中被软化，在 v3 中被模糊引用，到 v5 时完全消失。

让我们运行一个对照实验：
1. 在前 5 轮植入 **5 个特定事实**。
2. 继续进行 **15 轮填充对话**（触发多次重新摘要）。
3. 每次摘要周期后，检查每个原始事实是否仍然出现在摘要中。
4. 可视化事实保留率如何随摘要版本退化。

In [ ]:
import re

FACTS = [
    ("我的全名是 Elena Vasquez。", "elena vasquez"),
    ("我出生于 1992 年 3 月 15 日。", "1992"),
    ("我有三只猫，分别叫 Mochi、Tofu 和 Tempeh。", "tempeh"),
    ("我的年薪是 $145,000。", "145"),
    ("我对草莓过敏。", "strawberr"),
]

FILLER_MESSAGES = [
    "有什么好的意面 carbonara 食谱？",
    "告诉我关于爵士乐的历史。",
    "黑洞是如何形成的？",
    "Python 和 JavaScript 的主要区别是什么？",
    "你能简单解释量子纠缠吗？",
    "世界上最高的建筑是什么？",
    "内燃机是如何工作的？",
    "有什么改善睡眠的建议？",
    "告诉我关于帝王蝶的迁徙模式。",
    "北极光是什么引起的？",
    "疫苗是如何工作的？",
    "海洋最深的地方在哪里？",
    "告诉我关于国际象棋的历史。",
    "GPS 是如何工作的？",
    "质数在密码学中有什么用途？",
]

# 使用小缓冲区运行实验以强制多次摘要周期
mem_drift = SummaryMemory(
    max_buffer_size=4,  # 每 2 轮触发一次摘要
    system_prompt="你是一个乐于助人的助手。用 1-2 句话简洁回复。",
)

# 阶段 1：植入事实
print("阶段 1：植入事实...")
for fact_text, _ in FACTS:
    mem_drift.chat(fact_text)
print(f"  已植入 {len(FACTS)} 个事实。目前摘要版本数：{len(mem_drift.summary_history)}")

# 阶段 2：填充对话以触发多次重新摘要
print("\n阶段 2：填充轮次（触发重新摘要）...")
for i, filler in enumerate(FILLER_MESSAGES):
    mem_drift.chat(filler)

print(f"  已完成 {len(FILLER_MESSAGES)} 轮填充对话。")
print(f"  总摘要版本数：{len(mem_drift.summary_history)}")
print(f"  总 LLM 摘要调用次数：{mem_drift._summary_calls}")

现在我们检查每个摘要版本中哪些事实得以保留。我们在每个摘要中搜索关键词（如"elena vasquez"或"1992"）。勾号表示事实存在。叉号表示在压缩过程中丢失。

In [ ]:
# 检查每个摘要版本中哪些事实得以保留
fact_labels = [f[0][:35] + "..." for f in FACTS]
fact_keywords = [f[1] for f in FACTS]

retention_matrix = []
for summary in mem_drift.summary_history:
    row = []
    for keyword in fact_keywords:
        present = keyword.lower() in summary.lower()
        row.append(1 if present else 0)
    retention_matrix.append(row)

# 打印保留表
print("各摘要版本中的事实保留情况")
print("=" * 70)
header = f"{'版本':<10}" + "".join(f"{label:<12}" for label in ["Elena V.", "生于1992", "3只猫", "$145K", "草莓过敏"])
print(header)
print("-" * 70)
for i, row in enumerate(retention_matrix):
    cells_str = "".join(f"{'  ✓':<12}" if v else f"{'  ✗':<12}" for v in row)
    print(f"v{i+1:<9}{cells_str}")

total_facts = len(FACTS)
final_retained = sum(retention_matrix[-1]) if retention_matrix else 0
print(f"\n最终保留：{final_retained}/{total_facts} 个事实存活到最后一个摘要版本。")

跨摘要版本可视化事实保留率。热力图（左）显示每个版本中哪些事实得以保留。柱状图（右）显示保留事实的总数。观察随着摘要周期累积，数字如何下降。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

versions = list(range(1, len(retention_matrix) + 1))
fact_names = ["Elena V.", "生于1992", "3只猫", "$145K 薪资", "草莓过敏"]

# 左：事实保留热力图
retention_arr = np.array(retention_matrix)
im = ax1.imshow(retention_arr.T, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
ax1.set_xticks(range(len(versions)))
ax1.set_xticklabels([f"v{v}" for v in versions], fontsize=8)
ax1.set_yticks(range(len(fact_names)))
ax1.set_yticklabels(fact_names)
ax1.set_xlabel("摘要版本")
ax1.set_title("事实保留热力图\n（绿色 = 存在，红色 = 丢失）")

# 添加文本标注
for i in range(len(versions)):
    for j in range(len(fact_names)):
        symbol = "✓" if retention_arr[i, j] else "✗"
        ax1.text(i, j, symbol, ha="center", va="center",
                 color="white" if retention_arr[i, j] == 0 else "black", fontsize=10)

# 右：每个版本保留的事实总数
totals = [sum(row) for row in retention_matrix]
colors = ["#22c55e" if t >= 4 else "#f59e0b" if t >= 2 else "#ef4444" for t in totals]
ax2.bar(range(len(versions)), totals, color=colors, alpha=0.85)
ax2.set_xticks(range(len(versions)))
ax2.set_xticklabels([f"v{v}" for v in versions])
ax2.set_ylabel("保留的事实数")
ax2.set_ylim(0, len(FACTS) + 0.5)
ax2.set_xlabel("摘要版本")
ax2.set_title("每个摘要版本保留的事实总数")
ax2.axhline(y=len(FACTS), color="gray", linestyle="--", alpha=0.3, label=f"全部 {len(FACTS)} 个事实")
ax2.legend()

for i, t in enumerate(totals):
    ax2.text(i, t + 0.15, str(t), ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("summary_drift.png", dpi=150, bbox_inches="tight")
plt.show()

print("摘要漂移显而易见：随着摘要周期累积，事实逐渐退化。")

打印每个摘要版本的完整文本。按顺序阅读，你可以看到语言如何变化以及细节如何丢失。这就是摘要漂移的实际表现。

In [ ]:
print("=== 完整摘要演化 ===\n")
for i, s in enumerate(mem_drift.summary_history):
    retained = sum(retention_matrix[i])
    print(f"--- 版本 {i+1}（{retained}/{len(FACTS)} 个事实保留）---")
    print(s)
    print()

## Token 成本：摘要记忆 vs. 全缓冲 vs. 滑动窗口

摘要记忆介于全缓冲（无限回忆，二次成本）和滑动窗口（硬截断，恒定成本）之间。让我们比较这三种方法。

In [ ]:
import matplotlib.pyplot as plt

# 模拟 30 轮的 token 增长
# 假设：每条消息 ~50 token，系统提示词 ~30 token，摘要 ~100 token
MSG_TOKENS = 50
SYS_TOKENS = 30
SUMMARY_TOKENS = 100
NUM_TURNS = 30
WINDOW_K = 10
BUFFER_SIZE = 6  # 每 3 轮摘要一次

buffer_cost, window_cost, summary_cost = [], [], []

for turn in range(1, NUM_TURNS + 1):
    n_msgs = turn * 2

    # 全缓冲：所有消息
    buffer_cost.append(SYS_TOKENS + n_msgs * MSG_TOKENS)

    # 滑动窗口：限制在 k 条
    window_cost.append(SYS_TOKENS + min(n_msgs, WINDOW_K) * MSG_TOKENS)

    # 摘要记忆：摘要 + 缓冲区（缓冲区定期重置）
    buffer_msgs = (n_msgs % BUFFER_SIZE) or BUFFER_SIZE
    summary_cost.append(SYS_TOKENS + SUMMARY_TOKENS + buffer_msgs * MSG_TOKENS)

turns = list(range(1, NUM_TURNS + 1))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(turns, buffer_cost, "o-", color="#ef4444", label="全缓冲", linewidth=2, markersize=4)
ax.plot(turns, window_cost, "s-", color="#22c55e", label=f"滑动窗口 (k={WINDOW_K})", linewidth=2, markersize=4)
ax.plot(turns, summary_cost, "^-", color="#6366f1", label=f"摘要记忆 (buf={BUFFER_SIZE})", linewidth=2, markersize=4)

ax.set_xlabel("对话轮次")
ax.set_ylabel("每次 API 调用的输入 Token")
ax.set_title("每轮输入 Token：三种记忆策略")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("cost_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"在第 {NUM_TURNS} 轮：")
print(f"  全缓冲:    {buffer_cost[-1]:,} token")
print(f"  滑动窗口: {window_cost[-1]:,} token")
print(f"  摘要记忆: {summary_cost[-1]:,} token")
print(f"\n注意：摘要记忆还会有额外的摘要调用成本（未在图中显示）。")

## 回忆测试：Agent 还能使用其摘要吗？

让我们验证 Agent 是否真的能利用摘要中的事实来回答问题。记住：它不再拥有原始消息。

In [ ]:
RECALL_QUESTIONS = [
    ("我的全名是什么？", "elena"),
    ("我是什么时候出生的？", "1992"),
    ("我的猫叫什么名字？", "mochi"),
    ("我的薪资是多少？", "145"),
    ("我对什么食物过敏？", "strawberr"),
]

print("=== 回忆测试（共 20 轮之后）===\n")
results = []
for question, keyword in RECALL_QUESTIONS:
    answer = mem_drift.chat(question)
    recalled = keyword.lower() in answer.lower()
    status = "✓ 记住了" if recalled else "✗ 遗忘了"
    results.append(recalled)
    print(f"  问题: {question}")
    print(f"  回答: {answer[:120]}")
    print(f"  {status}")
    print()

score = sum(results)
print(f"得分: {score}/{len(RECALL_QUESTIONS)} 个事实从摘要中回忆出来。")

## 缓解摘要漂移

摘要漂移是本技术固有的，但有几种策略可以减少其影响：

### 1. 更好的摘要提示词
明确说明要保留什么：
```
保留对话中的所有以下内容：
- 专有名词（名称、地点、组织）
- 数字（日期、金额、数量）
- 陈述的偏好和约束
- 做出的决策及其理由
```

### 2. 结构化摘要
使用结构化格式而非自由形式文本：
```
事实: [name: Elena Vasquez, born: 1992-03-15, ...]
决策: [选择 Python 而非 Java, ...]
待解决问题: [预算尚未决定, ...]
```
这使 LLM 更难意外丢弃字段。

### 3. 更大的缓冲区大小
更少的摘要周期意味着更少的漂移。如果你的上下文窗口允许，使用 `max_buffer_size=20` 而非 `max_buffer_size=4`。

### 4. 摘要 + 缓冲混合方案
在摘要旁保留最后 *k* 条消息的原文。这是 LangChain 的 `ConversationSummaryBufferMemory` 等生产系统中使用的方法。我们在[技术 04](../04_summary_buffer_memory/) 中详述。

### 5. 实体提取
将关键实体提取到独立的存储中，并在摘要旁注入它们。即使摘要漂移，实体存储也能保留结构化事实。

### 6. 定期完全重新摘要
不要总是增量合并，偶尔从更大块的原始历史中重新摘要。这减少了摘要链中的复合误差。

## 讨论与权衡

### 优势
- **无界对话**：与滑动窗口不同，摘要记忆可以处理任意长的对话而不会丢失所有旧上下文。
- **有界的 token 成本**：摘要很紧凑。每轮成本大致恒定（摘要 + 缓冲区）。
- **优雅降级**：信息是逐渐压缩的，而不是硬截断。重要主题往往比次要细节保留更久。
- **灵活的压缩**：摘要提示词控制保留什么。你可以针对你的领域进行调整。

### 劣势
- **摘要漂移**：重复压缩引入累积信息丢失。具体数字、名称和细节最易受损。
- **额外的 LLM 调用**：每个摘要周期消耗 token 并增加延迟。使用激进的缓冲区大小 4 时，你每 2 轮就要进行一次额外的 LLM 调用。
- **非确定性记忆**：同一对话的两次运行可能产生不同的摘要，导致不同的 Agent 行为。
- **难以调试**：当 Agent "遗忘"某事时，很难判断是摘要器丢弃了它还是聊天模型忽略了它。
- **无法精确回忆**：Agent 永远无法从摘要历史中逐字引用你。

### 何时使用摘要记忆

| 场景 | 建议 |
|----------|---------------|
| 长对话（50+ 轮） | 很合适。在保留大意同时控制成本。 |
| 需要精确回忆早期事实 | 不理想。改用缓冲或检索增强记忆。 |
| 成本敏感型应用 | 很合适。比全缓冲便宜得多。 |
| 短对话（10 轮以下） | 开销不值得。使用缓冲或窗口。 |
| 多会话 Agent | 很合适。摘要可以轻松在会话之间传递。 |

### 成本模型
对于 *n* 轮对话，缓冲区大小为 *b*：
- **聊天调用：** *n*（与任何方案相同）
- **摘要调用：** floor(*n* / (*b*/2))（每次缓冲区刷新一次）
- **总额外成本：** 与 *n/b* 成正比。当 *b* 合理时，这只占总成本的一小部分。

## 进一步阅读

- [LangChain ConversationSummaryMemory](https://python.langchain.com/docs/modules/memory/types/summary?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：带有可自定义提示词的生产级实现
- [LangChain ConversationSummaryBufferMemory](https://python.langchain.com/docs/modules/memory/types/summary_buffer?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：将摘要与 token 限制缓冲区结合的混合方案
- [Anthropic: Building Conversational AI](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：官方多轮对话模式
- [Recursive Summarization with LLMs](https://arxiv.org/abs/2301.13848)：关于迭代摘要质量的研究
- [Lilian Weng, "LLM Powered Autonomous Agents"](https://lilianweng.github.io/posts/2023-06-23-agent/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：包含记忆架构的概述

---

*← 上一章：02 滑动窗口记忆 · 下一章：[04 摘要缓冲记忆](../04_summary_buffer_memory/) →*

In [ ]:
# 清理演示过程中创建的临时文件
import os
for f in ["summary_drift.png", "cost_comparison.png"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"已删除 {f}")

## 🧪 自己动手试试

三个小挑战来加深你的理解。每个应该花费 10-30 分钟。

### 挑战 1：摘要提示词工程
编写三个不同的 `SUMMARIZER_PROMPT` 变体：一个优先保留事实，一个优先保留用户偏好，一个优先保留行动项。对每个变体运行相同的 10 轮对话，并排比较生成的摘要。

### 挑战 2：量化摘要漂移
将漂移实验扩展到 30 轮。每次摘要后，使用 `extract_topic_tags()` 捕获主题。绘制随时间变化的幸存原始轮次主题数量。计算主题在摘要中的半衰期。

### 挑战 3：双层摘要
维护两个摘要：一个详细摘要（最近 10 轮）和一个高层摘要（之前的所有内容）。当详细摘要超出阈值时，将其压缩到高层摘要中。这反映了 15 记忆压缩中的模式。

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--03-summary-memory--summary-memory)
